# --- PASO 0: C4RG4 DE D4TOS  ----
B4se de d4tos s4c4d4s de l4 unific4ción de los 12 fomrul4rios. 

### df_resultados_dict
L4 b4se de d4tos de los result4dos contiene l4s column4s:
- ID_sujeto (Suj_001...)
- origen_form (Respuestas de formulario 1...)
- identidad (ABC...)
- bloque (Bloque_1...)
- dilema (Bloque_CON...)
- categoria (1...)
- respuesta (Opción 2...)
contiene 21 fil4s por c4d4 sujeto

### df_demograficos_dict
L4 b4se de d4tos de los cuestion4rios contiene l4s column4s:
- ID_Sujeto (Suj_001...)
- Origen_Form (Respuestas de formulario 1...)
- expectativa_sin, expectativa_grande, expectativa_pequeña (pregunt4s de expect4tiv4)
- ndc_1, ndc_2,	ndc_3,	ndc_4,	ndc_5,	ndc_6 (need for cognition)
- sdo_1, sdo_2,	sdo_3,	sdo_4,	sdo_5,	sdo_6,	sdo_7, sdo_8, sdo_9, sdo_10 (soci4l domin4nce orient4tion)
- Genero 
- politica 
- nivel_se

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import chi2_contingency

In [2]:
# 1. Cargar las bases de datos
df_res_dict = pd.read_csv('Base_res_dict.csv')
df_dem_dict = pd.read_csv('Base_Dem_dict.csv')
df_res_coop = pd.read_csv('Base_res_coop.csv')
df_info_bloques = pd.read_csv('Inform4ción blo#ues.csv')

# 2. Pre-procesamiento de Info Bloques para el merge
# Renombramos 'Nombre' a 'Bloque' para que coincida con df_res_dict
df_info_bloques = df_info_bloques.rename(columns={'Nombre': 'Bloque'})

# Seleccionamos variables de interés (Gap)
# Usamos 'Diferencia desigualdad final entre opciones' como el costo de revertir
cols_interes = ['Bloque', 'Diferencia desigualdad final entre opciones']
df_info_bloques_sel = df_info_bloques[cols_interes]

print("Datos cargados correctamente.")
print(f"Sujetos totales: {df_res_dict['ID_Sujeto'].nunique()}")

Datos cargados correctamente.
Sujetos totales: 166


## --- PASO 1: LIMPIEZA  ---
-Se eliminan los sujetos que fallaron las preguntas de atención (con fallar 1 chequeo de atención ya se elimina al sujeto)
Recibe:
- df_demograficos_dict
- df_resultados_dict

162 sujetos

- df_resultados_coop
- df_demograficos_coop

46 sujetos


Devuelve:
- df_res_dict_clean
- df_demo_dict_clean

121 sujetos

- df_res_coop_clean
- df_demo_coop_clean

42 sujetos

## --- PASO 2: C4mbio de v4ri4bles  ---
- Se cre4 l4 v4ri4ble "Mantiene" (v4le 1 se responde opcion 1)
- Se c4mbi4 l4 v4ri4ble expect4tiv4 (v4le 1, 0, -1; h4y efecto positivo en opc1, no lo h4y, h4y efecto positivo en opc2)
- Se cre4 Scores Psicométricos (SDO y NDC)
- Se cre4 df_master (Unimos Resultados + Demográficos + Info Bloques )
- se cre4 column4 de expect4tiv4s_num (#ue respondió c4d4 sujeto en c4d4 g4p)
- se c4mbi4 nombre de column4
-     "Diferencia desigualdad final entre opciones" --> Gap_Size

In [3]:
# ==============================================================================
# cre4ción de column4 expect4tiv4 4ctiv4
# ==============================================================================

def asignar_expectativa_segura(row):
    bloque = row['Bloque']
    
    # Bloques Sin Gap
    if bloque in ["Bloque_1", "Bloque_3"]:
        return row.get('expectativa_sin_num', np.nan)
        
    # Bloques Gap Grande
    elif bloque in ["Bloque_5", "Bloque_7"]:
        return row.get('expectativa_grande_num', np.nan)
        
    # Bloques Gap Pequeño
    elif bloque in ["Bloque_9", "Bloque_11"]:
        return row.get('expectativa_pequeña_num', np.nan)
        
    else:
        return np.nan

In [4]:
# 1. Codificar Variable Dependiente: REVERSIÓN DE RANKING
# Opción 2 = Revierte el ranking -> Codificamos como 0 (Reversión)
# Opción 1 = Mantiene el ranking -> Codificamos como 1 (Mantiene)
print("Conteo Original:")
print(df_res_dict['Respuesta'].value_counts())

df_res_dict['Mantiene'] = df_res_dict['Respuesta'].apply(lambda x: 1 if x == 'Opción 1' else 0)

print("\nConteo Nueva Variable:")
print(df_res_dict['Mantiene'].value_counts())

Conteo Original:
Opción 2    1844
Opción 1    1226
Opción 3     415
Name: Respuesta, dtype: int64

Conteo Nueva Variable:
0    2259
1    1226
Name: Mantiene, dtype: int64


In [5]:
# 2. C4mbi4mos l4 sección de expect4tiv4s. Respuest4s y titulos p4r4 3ue se4 m4s sencillo
#respuest4s de expect4tiv4s. Definimos el diccionario con el mapeo exacto
mapeo_expectativas = {
    "No espero ningún efecto": 0,
    "La Opción 1 incrementa la probabilidad de cooperación": 1,
    "La opción 1 incrementa la posibilidad de cooperación" : 1,
    "La Opción 2 incrementa la probabilidad de cooperación": -1,
    "La opción 2 incrementa la posibilidad de cooperación": -1
}

cols_expectativas = ['expectativa_sin', 'expectativa_grande', 'expectativa_pequeña']

# Aplicamos el reemplazo
for col in cols_expectativas:
    if col in df_dem_dict.columns:
        df_dem_dict[col] = df_dem_dict[col].replace(mapeo_expectativas)

# CHE#UEO R4PIDO
#--------------------------------------------------------
## 1. list de column4s 4 prob4r
#cols_reales = ['expectativa_sin', 'expectativa_grande', 'expectativa_pequeña']

## 2. Verificamos columna por columna
#for col in cols_reales:
#    if col in df_dem_clean.columns:
#        valores = df_dem_clean[col].unique()
#        print(f"[{col}] Valores únicos: {valores}")
        
        # Alerta rápida si detecta texto
#        if df_dem_clean[col].dtype == 'object':
#             print(f"⚠️ ALERTA: La columna {col} todavía es texto. Revisa espacios o tildes.")
#    else:
#        print(f"❌ ERROR: No encuentro la columna '{col}'. Revisa si escribiste bien el nombre en la base de datos.")
#    print("-" * 30)
#-----------------------------------------------------------------

In [6]:
# 3. Calcular Scores Psicométricos (SDO y NDC)
# Definir grupos de columnas

sdo_cols = [c for c in df_dem_dict.columns if 'sdo_' in c]
ndc_cols = [c for c in df_dem_dict.columns if 'ndc_' in c]

# Ítems a invertir
items_inv_ndc = ['ndc_3', 'ndc_4']
items_inv_sdo = ['sdo_2', 'sdo_4', 'sdo_6', 'sdo_8', 'sdo_10']

# Aplicar inversión para NDC (Escala 1-7 -> n+1 = 8)
for col in items_inv_ndc:
    df_dem_dict[col] = 6 - df_dem_dict[col]

# Aplicar inversión para SDO (Escala 1-5 -> n+1 = 6)
for col in items_inv_sdo:
    df_dem_dict[col] = 6 - df_dem_dict[col]

# Calcular Scores Psicométricos (Promedios)
df_dem_dict['SDO_Score'] = df_dem_dict[sdo_cols].mean(axis=1)
df_dem_dict['NDC_Score'] = df_dem_dict[ndc_cols].mean(axis=1)


In [7]:
df_dem_dict['NDC_Score'].unique()

array([3.5       , 3.83333333, 4.33333333, 2.33333333, 3.16666667,
       3.33333333, 3.66666667, 4.16666667, 2.5       , 2.83333333,
       4.66666667, 4.5       , 4.        , 5.        , 2.66666667,
       3.        , 4.83333333, 1.5       ])

In [8]:
# 4. Merge Final (Unir todo en un Master DataFrame)
# Unimos Resultados + Demográficos + Info Bloques 
cols_demograficas = [
    'ID_Sujeto', 
    'SDO_Score', 
    'NDC_Score',
    'expectativa_sin',    
    'expectativa_grande',   
    'expectativa_pequeña',
    'Genero', 
    'politica',
    'nivel_se'
]

df_master_sucio = df_res_dict.merge(df_dem_dict[cols_demograficas], on='ID_Sujeto', how='left')
df_master_sucio = df_master_sucio.merge(df_info_bloques_sel, on='Bloque', how='left')

In [9]:
#df_master.columns.tolist()

In [10]:
# 5. Creo un4 column4 nuev4 sobre el df_m4ster
cols_exp = ['expectativa_sin', 'expectativa_grande', 'expectativa_pequeña']

for col in cols_exp:
    # Creamos las versiones numéricas: ej. 'expectativa_sin_num'
    if col in df_master_sucio.columns:
        df_master_sucio[f'{col}_num'] = df_master_sucio[col].fillna(0)
        

# Aplicamos la función
df_master_sucio['Expectativa_Activa'] = df_master_sucio.apply(asignar_expectativa_segura, axis=1)

In [11]:
# 6. Renombrar columna de Gap para facilitar uso
df_master_sucio.rename(columns={'Diferencia desigualdad final entre opciones': 'Gap_Size'}, inplace=True)


In [12]:
col_gap = 'Gap_Size' 

# Esto creará 5 columnas nuevas (una por cada tamaño de gap) para cada sujeto
df_gaps = df_master_sucio.pivot_table(index='ID_Sujeto', 
                         columns=col_gap, 
                         values='Mantiene', 
                         aggfunc='mean')

# 4. RENOMBRAR LAS NUEVAS COLUMNAS
df_gaps = df_gaps.add_prefix('Promedio_Gap_')

# Verificamos qué se creó
print("Nuevas variables creadas:", df_gaps.columns.tolist())

# 5. INTEGRACIÓN (MERGE) A LA BASE MAESTRA
# Reseteamos el índice para poder usar ID_Sujeto en el merge
df_gaps_reset = df_gaps.reset_index()

# Hacemos el cruce (Left Join)
# Esto pega las 5 columnas nuevas a la derecha de tu tabla gigante
df_master_sucio = pd.merge(df_master_sucio, df_gaps_reset, on='ID_Sujeto', how='left')

# ==============================================================================
# 6. GUARDAR
# ==============================================================================
print("\n--- Vista Previa (Primer sujeto) ---")
# Mostramos el ID y las nuevas columnas de Gap
cols_to_show = ['ID_Sujeto'] + df_gaps.columns.tolist()
print(df_master_sucio[cols_to_show].head(1))


Nuevas variables creadas: ['Promedio_Gap_0.0', 'Promedio_Gap_1000.0', 'Promedio_Gap_1200.0', 'Promedio_Gap_2000.0', 'Promedio_Gap_2400.0']

--- Vista Previa (Primer sujeto) ---
  ID_Sujeto  Promedio_Gap_0.0  Promedio_Gap_1000.0  Promedio_Gap_1200.0  \
0   Suj_001          0.666667             0.666667             0.666667   

   Promedio_Gap_2000.0  Promedio_Gap_2400.0  
0             0.333333                  0.0  


In [13]:
print("Variables creadas: Mantiene, SDO_Score, NDC_Score, Gap_Size, Expectativa_Activa")
print(df_master_sucio[['ID_Sujeto', 'Bloque','Dilema', 'Mantiene', 'Gap_Size', 'SDO_Score', 'Expectativa_Activa']].head())

Variables creadas: Mantiene, SDO_Score, NDC_Score, Gap_Size, Expectativa_Activa
  ID_Sujeto    Bloque      Dilema  Mantiene  Gap_Size  SDO_Score  \
0   Suj_001  Bloque_1  Bloque_CON         0       0.0        2.1   
1   Suj_001  Bloque_5  Bloque_CON         0    2000.0        2.1   
2   Suj_001  Bloque_7  Bloque_CON         0    2400.0        2.1   
3   Suj_001  Bloque_0    Atencion         0       NaN        2.1   
4   Suj_001  Bloque_3  Bloque_CON         0       0.0        2.1   

   Expectativa_Activa  
0                 0.0  
1                -1.0  
2                -1.0  
3                 NaN  
4                 0.0  


In [14]:
df_wide = df_master_sucio.pivot_table(index='ID_Sujeto', 
                         columns='Dilema', 
                         values='Mantiene', 
                         aggfunc='mean')

# 3. CÁLCULO DEL DELTA (VARIABLE DEPENDIENTE)
# Fórmula: Delta = Promedio(CON) - Promedio(SIN)  
# El Cálculo Matemático
df_wide['Promedio_CON'] = df_wide['Bloque_CON']
df_wide['Promedio_SIN'] = df_wide['Bloque_SIN']
df_wide['Promedio_DIST'] = df_wide['Dist']
df_wide['Delta_Mantiene'] = df_wide['Promedio_CON'] - df_wide['Promedio_SIN']
df_wide['Delta_base'] = df_wide['Promedio_SIN'] - df_wide['Promedio_DIST']
df_wide.head()

columnas_finales = ['Promedio_CON', 'Promedio_SIN', 'Promedio_DIST', 'Delta_Mantiene', 'Delta_base']
df_delta = df_wide[columnas_finales]

In [15]:
df_delta.head()
#df_delta.to_csv('df_delta.csv', index=False)

Dilema,Promedio_CON,Promedio_SIN,Promedio_DIST,Delta_Mantiene,Delta_base
ID_Sujeto,,,,,
Suj_001,0.166667,0.666667,0.666667,-0.500000,0.000000
Suj_002,0.333333,0.666667,0.500000,-0.333333,0.166667
Suj_003,0.333333,0.166667,0.166667,0.166667,0.000000
Suj_004,0.333333,0.166667,0.166667,0.166667,0.000000
Suj_005,0.333333,0.166667,0.333333,0.166667,-0.166667


In [16]:
df_long1 = pd.merge(df_master_sucio, df_delta, on='ID_Sujeto', how='left')
df_long1.head()

,ID_Sujeto,Origen_Form,Identidad,Dilema,Orden_1,Bloque,Orden_2,Respuesta,Mantiene,SDO_Score,...,Promedio_Gap_0.0,Promedio_Gap_1000.0,Promedio_Gap_1200.0,Promedio_Gap_2000.0,Promedio_Gap_2400.0,Promedio_CON,Promedio_SIN,Promedio_DIST,Delta_Mantiene,Delta_base
0,Suj_001,Respuestas de formulario 1,ABC,Bloque_CON,1,Bloque_1,1,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
1,Suj_001,Respuestas de formulario 1,AEF,Bloque_CON,1,Bloque_5,2,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
2,Suj_001,Respuestas de formulario 1,AJK,Bloque_CON,1,Bloque_7,3,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
3,Suj_001,Respuestas de formulario 1,-,Atencion,1,Bloque_0,1,Opción 3,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0
4,Suj_001,Respuestas de formulario 1,AGH,Bloque_CON,1,Bloque_3,4,Opción 2,0,2.1,...,0.666667,0.666667,0.666667,0.333333,0.0,0.166667,0.666667,0.666667,-0.5,0.0


In [17]:
import pandas as pd
import numpy as np

# 1. CARGA DE DATOS
# df_long = pd.read_csv('Base_Maestra_Completa.csv')
df_actual = df_long1.copy()

# ==============================================================================
# FASE 1: DESCUBRIR A QUÉ BLOQUE PERTENECE LA PREGUNTA DE ATENCIÓN
# ==============================================================================

# 1. Creamos una columna nueva copiando los dilemas
df_actual['Tratamiento_Real'] = df_actual['Dilema']

# 2. CORRECCIÓN: Detectamos las filas de atención SOLO mirando la columna Dilema.
# Así evitamos que ensayos normales con Identidad vacía se confundan con atención.
es_atencion = df_actual['Dilema'].astype(str).str.contains('Atencion|Atención', case=False, na=False)

# 3. Borramos el nombre en esas filas para que queden como vacíos (NaN)
df_actual.loc[es_atencion, 'Tratamiento_Real'] = np.nan

# 4. LA MAGIA: Rellenamos los vacíos copiando el nombre del tratamiento de la fila de arriba 
df_actual['Tratamiento_Real'] = df_actual.groupby('ID_Sujeto')['Tratamiento_Real'].ffill()

# 5. Creamos la llave maestra: Sujeto + Tratamiento Real
df_actual['Llave_Bloque'] = df_actual['ID_Sujeto'].astype(str) + "_" + df_actual['Tratamiento_Real'].astype(str)


# ==============================================================================
# FASE 2: DETECTAR FALLOS DE ATENCIÓN (ROBUSTO)
# ==============================================================================

# A. Aislar las preguntas de atención 
filas_atencion = df_actual[es_atencion]

# B. CORRECCIÓN: Encontrar las que fallaron. 
# En lugar de buscar texto exacto, buscamos si la respuesta NO (~) contiene un "3"
fallos = filas_atencion[~filas_atencion['Respuesta'].astype(str).str.contains('3')]

# C. Listas negras
llaves_bloques_fallados = fallos['Llave_Bloque'].unique()
ids_sujetos_distraidos_total = fallos['ID_Sujeto'].unique()

# Géneros inválidos
generos_validos = ["Mujer", "Hombre"]
ids_genero_invalido = df_actual[~df_actual['Genero'].isin(generos_validos)]['ID_Sujeto'].unique()
# ==============================================================================
# FASE 3: BORRADO DE LA PREGUNTA Y CREACIÓN DE BASES
# ==============================================================================
# Eliminamos las filas de atención originales
df_madre = df_actual[~es_atencion].copy()

# df_long1: Base intacta
df_long1 = df_madre.copy()

# df_long2: Excluye géneros inválidos
df_long2 = df_madre[~df_madre['ID_Sujeto'].isin(ids_genero_invalido)].copy()

# df_long3: Excluye SOLO EL TRATAMIENTO donde se distrajeron
df_long3 = df_madre[~df_madre['Llave_Bloque'].isin(llaves_bloques_fallados)].copy()

# df_long4: Excluye TRATAMIENTO distraído Y géneros inválidos
df_long4 = df_madre[
    (~df_madre['Llave_Bloque'].isin(llaves_bloques_fallados)) & 
    (~df_madre['ID_Sujeto'].isin(ids_genero_invalido))
].copy()

# df_long5: Exclusión TOTAL de sujetos distraídos Y géneros inválidos
df_long5 = df_madre[
    (~df_madre['ID_Sujeto'].isin(ids_sujetos_distraidos_total)) & 
    (~df_madre['ID_Sujeto'].isin(ids_genero_invalido))
].copy()


# ==============================================================================
# FASE 4: REPORTE Y GUARDADO
# ==============================================================================

bases = {
    'df_long0':df_actual,
    'df_long1_Completa': df_long1,
    'df_long2_FiltroGenero': df_long2,
    'df_long3_FiltroBloque': df_long3,
    'df_long4_FiltroBloque_Y_Genero': df_long4,
    'df_long5_FiltroSujetoTotal_Y_Genero': df_long5
}

print("="*55)
print("📊 REPORTE DE LIMPIEZA DE DATOS")
print("="*55)

for nombre, df in bases.items():
    # Limpieza final de columnas auxiliares
    columnas_a_borrar = [col for col in ['Llave_Bloque', 'Tratamiento_Real'] if col in df.columns]
    df = df.drop(columns=columnas_a_borrar)
    
    sujetos = df['ID_Sujeto'].nunique()
    ensayos = len(df)
    
    print(f"📄 {nombre}:")
    print(f"   -> Sujetos (n) = {sujetos}")
    print(f"   -> Ensayos     = {ensayos}\n")
    
    # Guardado
    df.to_csv(f"{nombre}.csv", index=False)

print("✅ Proceso terminado. Las 6 bases están listas y guardadas.")

📊 REPORTE DE LIMPIEZA DE DATOS
📄 df_long0:
   -> Sujetos (n) = 166
   -> Ensayos     = 3485

📄 df_long1_Completa:
   -> Sujetos (n) = 166
   -> Ensayos     = 2987

📄 df_long2_FiltroGenero:
   -> Sujetos (n) = 161
   -> Ensayos     = 2897

📄 df_long3_FiltroBloque:
   -> Sujetos (n) = 152
   -> Ensayos     = 2490

📄 df_long4_FiltroBloque_Y_Genero:
   -> Sujetos (n) = 148
   -> Ensayos     = 2424

📄 df_long5_FiltroSujetoTotal_Y_Genero:
   -> Sujetos (n) = 121
   -> Ensayos     = 2178

✅ Proceso terminado. Las 6 bases están listas y guardadas.
